# Gradient boosting trees
We have explored several machine learning algorithms, from decision trees to support vector machines. Decision trees are particularly popular because of their simplicity and interpretability. However, individual trees can easily overfit the training data and may struggle to capture complex relationships without becoming excessively deep.

Ensemble methods address some of these limitations by combining multiple models. Random forests and extremely randomized trees, for example, combine many decision trees to build a more robust estimator. This idea of combining multiple relatively simple models into a stronger one is known as ensemble learning.

In a previous post, we introduced gradient descent in the context of neural networks: an optimization technique that iteratively updates model parameters in the direction that reduces the loss function. Gradient boosting combines a similar optimization idea with decision trees.

## Boosting
Boosting is an ensemble technique that builds a strong model by combining multiple weak learners sequentially.

This is different from bagging, where individual learners are trained independently—often on different samples of the training data—and their predictions are then aggregated.

In boosting, models are trained sequentially. Each new learner is added to improve the predictions made by the ensemble built so far. In other words, instead of training every tree independently, boosting uses the information from previous iterations to determine what the next tree should learn.

The final model can be viewed as a weighted sum of the predictions produced by the individual learners.

## Gradient boosting decision trees
Gradient boosting decision trees is a ML strategy that uses a boosting approach with the descent gradient algorithm to optimize model performance on decision tree estimators. Formerly the model is defined as:

$$
    F(x) = F_{0}(x) + \eta \sum_{m=1}^{M} h_{m}(x)
$$

where:
- $F_{0}(x)$ is the initial model or base prediction.
- $h_{m}(x)$ is the decision tree added at iteration $m$.
- $M$ is the total number of boosting iterations (trees).
- $\eta$ is the learning rate, whic controls the contribution of each new tree.

The key idea is that each new tree is trained to improve the current model by approximating the direction in which the loss function should decrease.

### Training
Unlike random forests, where trees are trained independently using bootstrap samples, gradient boosting builds trees sequentially. Each new tree attempts to correct the errors made by the ensemble constructed in the previous iterations.

**Algorithm:**

Consider a training set:
$$
    ((x_i, y_i))_{i=1}^n
$$

and a differentiable loss function $L(y, F(x))$.

1. Choose a base learner $h_0$ as a single decision tree that minimizes $L$ over the training data.
2. Compute the current predictions $F_0(x)$.
3. Calculate the pseudo-residuals $r_{k,i}$ for each sample $i$ in the base learner.

$$
r_{0,i} = -\frac{\partial L(y_i, F_{0}(x_i))}{\partial F_{0}(x_i)}
$$

4. Train a new estimator $h_{k+1}$ to approximate these negative gradients.

$$
    h_{1}(x_i) \approx r_{0,m}
$$

5. Add the new tree to the existing model:

$$
    F_{k}(x) = F_{k-1}(x) + \eta h_k(x)
$$

6. Repeat steps 2-5 until reaching the desired number of trees or until stopping criterion is satisfied. In practice, early stopping is often used to stop training when performance on a validation set no longer improves.

## Libraries

### Scikit learn
Provides two main implementations of gradient boosting based on decision trees:

- `GradientBoostingClassifier` and `GradientBoostingRegressor`.
- `HistGradientBoostingClassifier` and `HistGradientBoostingRegressor`.

The standard `GradientBoosting*` implementations build trees using the traditional approach, where candidate split points for numerical features are considered from the observed values in the training data.

The `HistGradientBoosting*` estimators use a histogram-based approach. Before constructing the trees, continuous features are discretized into a fixed number of bins. Instead of evaluating split points for potentially many distinct feature values, the algorithm searches for splits between bins.

This significantly reduces the number of candidate splits and can make training considerably faster in big datasets, especially for large datasets. The default maximum number of bins is 255, although this is configurable through the `max_bins` parameter.

### XGBoost

XGBoost (Extreme Gradient Boosting) is a high-performance machine learning library built around gradient-boosted decision trees. Its core implementation is primarily written in C++, with interfaces available for languages such as Python and R.

XGBoost extends the standard gradient boosting framework with several features designed to improve performance, scalability, and generalization, including:

- Regularization to control model complexity.
- Efficient tree construction algorithms.
- Row and column subsampling.
- Support for parallel and distributed training.
- GPU acceleration.
- Missing-value handling.
- Support for categorical features in recent versions.
- Early stopping.
- Second-order optimization using both gradients and Hessians.

XGBoost provides an API that integrates well with the Scikit-learn ecosystem through classes such as `XGBClassifier` and `XGBRegressor`. Consequently, XGBoost models can be used with Scikit-learn utilities such as `Pipeline`, cross-validation, and hyperparameter search.

## How inferences are performed?

### Classification
Note that the value $F(x)$ is a continuos value, so it can't be a class label or probability score. We need to define a decision function for mapping the $F(x)$ values to a class labels or probability scores.

#### Binary classification
For binary classification, the model produces a single raw score $F(x)$. This score is commonly interpreted as a log-odds (logit) when using the logistic objective.
The sigmoid function converts the raw score into the probability of the positive class:
$$
    p(y = 1 \mid x) = \sigma(F(x)) = \frac{1}{1+e^{-F(x)}}
$$

A class prediction can then be obtained using a threshold, commonly $0.5$:

$$
    \hat{y}(x) =
    \begin{cases} 
        1 & \text{if } x \geq 0.5 \\ 
        0 & \text{otherwise}
    \end{cases}
$$

#### Multiclass classification
For multiclass classification, the model produces one raw score for each class. Suppose there are $K$ classes. The model produces:
$$
    F_1(x), F_2(x), ..., F_k(x)
$$

These raw scores can be converted into a probability distribution using the softmax function:

$$
    p(y = k \mid x) = softmax(F(x))_{k} = \frac{e^{F_k(x)}}{\sum_{j=1}^{K} e^{F_j{x}}}
$$

The resulting probabilities sum to one and the predicted class is then the class with the highest probability:

$$
    \hat{y}(x) = \operatorname*{arg\,max}_{k \in \{1,\ldots, K \}}
p(y = k \mid x)
$$

It is important to note that the individual trees used inside the boosting process are still regression trees. They do not directly predict class labels. Instead, their outputs contribute to the raw scores that are subsequently transformed into probabilities.

In [ ]:
from pandas import read_csv
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = MLPClassifier(hidden_layer_sizes=(20, 15, 10, 5), solver="lbfgs", activation="relu", random_state=123)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

### Regression
For regression, no additional probability or decision function is required. The model directly produces a continuous prediction:

$$
    \hat{y}(x)=F(x)
$$

Regression can use the raw model output directly, while classification generally requires a link function such as the sigmoid or softmax to transform raw scores into probabilities

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = MLPRegressor(hidden_layer_sizes=(8,8,8,4,2), activation="identity", solver="lbfgs", random_state=123)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")